In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import re

In [2]:
file_path = "合并324份2.17.xlsx" 

print("正在加载数据...")
try:
    df_raw = pd.read_excel(file_path)
    print(f"读取成功，原始样本量: {len(df_raw)}")
except FileNotFoundError:
    print("❌ 错误：找不到文件，请检查文件名是否正确。")

正在加载数据...
读取成功，原始样本量: 323


In [9]:
'''
定位陷阱题列
判断数值是否为1
包含字符和数值型
'''
# 1. 加载数据
file_path = "合并324份2.17.xlsx"
df_raw = pd.read_excel(file_path)
# 2. 定位陷阱题Q5
trap_cols = [c for c in df_raw.columns if 'Q5' in c and '显示测试' in c]
if not trap_cols:
    print("未找到陷阱题列，请检查列名。")
else:
    trap_col = trap_cols[0]
    total_count = len(df_raw)
    eliminated_mask = (df_raw[trap_col] == 1) | (df_raw[trap_col] == 1.0)
    valid_df = df_raw[~eliminated_mask].copy()
    
    # 5. 计算剔除率
    eliminated_count = eliminated_mask.sum()
    drop_rate = eliminated_count / total_count

    print(f"--- 问卷清洗结果 ---")
    print(f"原始总样本量: {total_count}")
    print(f"被淘汰人数 (选B/数值1): {eliminated_count}")
    print(f"保留样本量 (包含0及其它): {len(valid_df)}")
    print(f"最终剔除率: {drop_rate:.2%}")

--- 问卷清洗结果 ---
原始总样本量: 323
被淘汰人数 (选B/数值1): 104
保留样本量 (包含0及其它): 219
最终剔除率: 32.20%


In [10]:
def parse_attributes(text, is_none=False):
    """
    修正版：移除通用关键词，防止属性间干扰。
    只匹配策划案中定义的独有中文描述。
    """
    if is_none:
        return {'Smart': 0, 'Context': 0, 'Privacy': 0, 'Price': 0, 'ASC': 0}
    
    # 预处理：统一转为字符串并清理
    text = str(text).replace('：', ':').replace(' ', '')
    
    # --- A1: 智能水平解析 ---
    # 关键词来源：策划案表3-1
    # L3: 专家创作 / 极少修改
    # L2: 进阶推理 / 准确率高
    # L1: 基础辅助 / 需核查 (Default)
    if any(k in text for k in ['专家', '极少修改']):
        smart = 3
    elif any(k in text for k in ['进阶', '推理', '准确率']):
        smart = 2
    else:
        # 如果找不到L2/L3的中文，尝试匹配带前缀的编码（防止串台）
        if '智能:L3' in text or '智能L3' in text: smart = 3
        elif '智能:L2' in text or '智能L2' in text: smart = 2
        else: smart = 1

    # --- A2: 上下文解析 ---
    # L3: 全资料库 / 长期记忆 / 超长
    # L2: 500页 / 整本 / 较长
    # L1: 标准 / 50页 (Default)
    if any(k in text for k in ['全资料', '长期记忆', '超长']):
        context = 3
    elif any(k in text for k in ['整本', '500页', '较长']):
        context = 2
    else:
        if '上下文:L3' in text or '上下文L3' in text: context = 3
        elif '上下文:L2' in text or '上下文L2' in text: context = 2
        else: context = 1

    # --- A3: 隐私解析 ---
    # L2: 严格保密 / 不留存 / 隐私模式
    # L1: 默认开启 (Default)
    if any(k in text for k in ['严格保密', '不留存', '隐私模式']):
        privacy = 2
    else:
        if '隐私:L2' in text or '隐私L2' in text: privacy = 2
        else: privacy = 1

    # --- A4: 价格解析 ---
    try:
        # 匹配数字，但排除掉像 "500页" 里的500
        # 逻辑：找后面跟着"元"的数字，或者位于字符串末尾的数字
        # 这里的正则策略：提取所有数字，看哪个像价格 (20, 60, 130)
        nums = re.findall(r'(\d+)', text)
        price = 0
        for n in nums:
            v = int(n)
            if v in [20, 60, 130, 19, 29, 39, 59, 99]: # 包含可能的实际标价
                price = v
                break
        # 如果没匹配到常见价格，取最后一个数字尝试（通常价格在最后）
        if price == 0 and nums:
            price = int(nums[-1])
    except:
        price = 0
    
    return {'Smart': smart, 'Context': context, 'Privacy': privacy, 'Price': price, 'ASC': 1}

In [11]:
print("\n正在重构数据结构 (Wide -> Long)...")
long_data = []

# 找到所有包含 "方案A" 的列（排除陷阱题）
choice_cols = [c for c in df_raw.columns if '方案A' in c and 'Q5' not in c and '显示测试' not in c]

for idx, row in valid_df.iterrows():
    respondent_id = row.get('序号', idx) # 如果没有序号列，用索引
    
    for q_col in choice_cols:
        # 提取题干中的文本
        # 假设题干格式："...方案A：【智能...】...方案B：【智能...】..."
        try:
            # 分割方案A和方案B的描述文本
            # 注意：这里需要根据您实际的 Excel 表头格式调整 split 关键字
            parts = str(q_col).split('方案B')
            text_a = parts[0].split('方案A')[-1] # 取方案A后面的部分
            text_b = parts[1]
        except IndexError:
            continue 
            
        user_choice = str(row[q_col])
        
        # 定义三个选项
        options = [
            ('A', text_a, False),
            ('B', text_b, False),
            ('None', '', True)
        ]
        
        for label, text, is_none_flag in options:
            # 解析属性
            attrs = parse_attributes(text, is_none=is_none_flag)
            
            # 判断是否被选中
            # 问卷星通常记录为 "方案A" 或 "方案B" 或 "我都不选"
            is_chosen = 0
            if label == 'A' and '方案A' in user_choice: is_chosen = 1
            elif label == 'B' and '方案B' in user_choice: is_chosen = 1
            elif label == 'None' and ('都不选' in user_choice or '基础版' in user_choice): is_chosen = 1
            
            long_data.append({
                'ID': respondent_id,
                'Q_ID': q_col[:10],
                'Choice': is_chosen,
                **attrs
            })

df_long = pd.DataFrame(long_data)
print("\n--- 数据解析质量检查 ---")
print("如果以下某一行只显示 1 或 0，说明关键词匹配失败，需要调整 parse_attributes 函数")
print("智能水平分布:\n", df_long[df_long['ASC']==1]['Smart'].value_counts().sort_index())
print("上下文分布:\n", df_long[df_long['ASC']==1]['Context'].value_counts().sort_index())
print("隐私分布:\n", df_long[df_long['ASC']==1]['Privacy'].value_counts().sort_index())


正在重构数据结构 (Wide -> Long)...

--- 数据解析质量检查 ---
如果以下某一行只显示 1 或 0，说明关键词匹配失败，需要调整 parse_attributes 函数
智能水平分布:
 Smart
1     657
2    1314
3    1971
Name: count, dtype: int64
上下文分布:
 Context
1    1314
2    1314
3    1314
Name: count, dtype: int64
隐私分布:
 Privacy
1    1971
2    1971
Name: count, dtype: int64


In [12]:
df_model = df_long.copy()

# 手动创建 Dummy 变量 (Reference Level = 1)
# 只要 Smart=2，Smart_2就为1，否则为0
df_model['Smart_2'] = (df_model['Smart'] == 2).astype(int)
df_model['Smart_3'] = (df_model['Smart'] == 3).astype(int)

df_model['Context_2'] = (df_model['Context'] == 2).astype(int)
df_model['Context_3'] = (df_model['Context'] == 3).astype(int)

df_model['Privacy_2'] = (df_model['Privacy'] == 2).astype(int)

# 定义自变量
X_cols = ['ASC', 'Price', 'Smart_2', 'Smart_3', 'Context_2', 'Context_3', 'Privacy_2']
y = df_model['Choice']
X = df_model[X_cols]

# ==========================================
# 7. 模型拟合 (Logit)
# ==========================================
print("\n正在拟合 Logit 模型...")
try:
    logit_model = sm.Logit(y, X)
    result = logit_model.fit(disp=0)
    
    print(result.summary())
    
    # ==========================================
    # 8. 支付意愿 (WTP) 计算
    # ==========================================
    print("\n" + "="*30)
    print("      支付意愿 (WTP) 分析")
    print("="*30)
    
    beta_price = result.params['Price']
    
    if beta_price >= 0:
        print("⚠️ 严重警告：价格系数为正 (Beta_Price > 0)，这意味着越贵用户越买。")
        print("可能原因：样本量太小、数据定义反了、或者用户没认真答题。WTP 计算将失效。")
    else:
        # WTP = - (Beta_Attribute / Beta_Price)
        def print_wtp(attr_name, desc):
            if attr_name in result.params:
                beta_attr = result.params[attr_name]
                wtp = - (beta_attr / beta_price)
                print(f"【{desc}】 WTP = {wtp:.2f} 元/月")
            else:
                print(f"❌ 无法计算 {desc} (变量被剔除)")

        print("--- 智能属性 (相对于基础辅助) ---")
        print_wtp('Smart_2', '进阶推理 (Level 2)')
        print_wtp('Smart_3', '专家创作 (Level 3)')
        
        print("\n--- 上下文属性 (相对于50页标准) ---")
        print_wtp('Context_2', '书籍级 (Level 2)')
        print_wtp('Context_3', '全资料库 (Level 3)')
        
        print("\n--- 隐私属性 (相对于默认开启) ---")
        print_wtp('Privacy_2', '严格保密 (Level 2)')

except Exception as e:
    print(f"模型运行出错: {e}")
    print("请检查上方‘数据解析质量检查’部分，确认是否存在某列全为0的情况。")


正在拟合 Logit 模型...
模型运行出错: Singular matrix
请检查上方‘数据解析质量检查’部分，确认是否存在某列全为0的情况。


C:\Users\liuyu\AppData\Roaming\Python\Python312\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))


In [13]:
import statsmodels.api as sm
from statsmodels.discrete.conditional_models import ConditionalLogit

# ==========================================
# 终极方案: Conditional Logit (CLogit)
# ==========================================
print("\n正在拟合 Conditional Logit 模型 (更严谨的方法)...")

# 1. 准备数据
# CLogit 需要数据按 Case (Q_ID) 分组
# 确保数据已经按 Q_ID 排序 (这就为什么我们在前面要生成 Q_ID)
df_model = df_model.sort_values(by=['ID', 'Q_ID'])

# 2. 定义变量
# 注意: CLogit 不需要 ASC 列，因为它比较的是组内差异
# 我们把 ASC 含义的变量改名为 "Buy" (购买意愿)，仅在 Option A/B 时为1，None时为0
# 这捕捉了"选择购买"相对于"都不选"的基准效用
df_model['Buy'] = df_model['ASC'] 

feature_cols = ['Buy', 'Price', 'Smart_2', 'Smart_3', 'Context_2', 'Context_3', 'Privacy_2']
y = df_model['Choice']
X = df_model[feature_cols]
groups = df_model['Q_ID'] # 告诉模型哪些行属于同一道题

try:
    # 3. 拟合模型
    clogit_model = ConditionalLogit(y, X, groups=groups)
    result_clogit = clogit_model.fit(disp=0)
    
    print(result_clogit.summary())
    
    # 4. 重新计算 WTP
    print("\n" + "="*30)
    print("      修正后的支付意愿 (CLogit WTP)")
    print("="*30)
    
    beta_price = result_clogit.params['Price']
    
    def print_wtp_c(attr, label):
        if attr in result_clogit.params:
            w = - result_clogit.params[attr] / beta_price
            print(f"【{label}】 WTP = {w:.2f} 元/月")
            
    print_wtp_c('Smart_2', '智能: 进阶 (L2)')
    print_wtp_c('Smart_3', '智能: 专家 (L3)')
    print_wtp_c('Privacy_2', '隐私: 严格保密')
    print_wtp_c('Buy', '品牌溢价/基础效用')

except Exception as e:
    print(f"CLogit 运行失败: {e}")
    print("提示: 确保 statsmodels 版本较新 (>=0.12)")


正在拟合 Conditional Logit 模型 (更严谨的方法)...
CLogit 运行失败: endog must be coded as 0, 1
提示: 确保 statsmodels 版本较新 (>=0.12)


C:\Users\liuyu\AppData\Roaming\Python\Python312\site-packages\statsmodels\discrete\conditional_models.py:80: UserWarning: Dropped 9 groups and 5913 observations for having no within-group variance
  warnings.warn(msg)
